# Паттерн 11: Round-Robin Discussion — обсуждение по кругу

Round-Robin — это совещание, где агенты по очереди дополняют общий результат. В отличие от Debate, здесь нет противостояния позиций — агенты сотрудничают, работая над одним документом. Каждый привносит свою экспертизу: структуру, факты, стиль. После прохода всех участников цикл повторяется — каждый новый раунд кумулятивно улучшает артефакт.

```mermaid
graph LR
    START((START)) --> structuralist(["🔹 Structuralist<br/><i>высказывается по очереди</i>"])
    structuralist --> factologist(["🔹 Factologist<br/><i>высказывается по очереди</i>"])
    factologist --> stylist(["🔹 Stylist<br/><i>высказывается по очереди</i>"])
    stylist --> counter(["🎯 Counter<br/><i>считает раунды</i>"])
    counter -->|next_round| structuralist
    counter -->|done| END((END))

    classDef terminal fill:#95A5A6,stroke:#707B7C,color:#fff
    classDef worker fill:#2ECC71,stroke:#1A8B4C,color:#fff,stroke-width:2px
    classDef coordinator fill:#4A90D9,stroke:#2C5F8A,color:#fff,stroke-width:2px

    class START,END terminal
    class structuralist,factologist,stylist worker
    class counter coordinator
```

In [1]:
import sys
sys.path.insert(0, "..")

from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage
from llm_config import get_llm, check_api_key

In [2]:
if not check_api_key():
    raise ValueError("API key is not set")
else:
    print("API key is set")

API key is set


## Состояние обсуждения

Состояние `RoundRobinState` содержит четыре поля. Ключевое — одно поле `document` (без reducer), которое каждый агент перезаписывает улучшенной версией. Это не накопление аргументов, а итеративная шлифовка одного артефакта. Поле `round` отслеживает текущий раунд, а `max_rounds` задаёт глубину обработки и контролирует стоимость.

In [3]:
llm = get_llm()

class RoundRobinState(TypedDict):
    task: str              # Исходная задача
    document: str          # Общий документ, который дополняют все агенты
    round: int             # Текущий раунд
    max_rounds: int        # Максимальное число раундов

## Три эксперта

Три агента с разными фокусами работают над одним документом:
- **Структуралист** — улучшает структуру, логику изложения, связность разделов. В первом раунде создаёт план с нуля, в последующих — реорганизует то, что наработали другие.
- **Фактолог** — проверяет факты, дополняет конкретными данными и примерами. Не меняет структуру — только обогащает содержание.
- **Стилист** — улучшает язык: убирает канцеляризмы, делает текст живым и читаемым. Сохраняет все факты и структуру.

In [4]:
def structuralist_node(state: RoundRobinState) -> dict:
    """Структуралист: улучшает структуру, логику, связность."""
    if not state["document"]:
        prompt = (
            f"Создай структурированный план документа по теме: {state['task']}. "
            f"Выдели 3-4 ключевых тезиса."
        )
    else:
        prompt = (
            f"Улучши структуру документа: проверь логику, "
            f"добавь недостающие тезисы, упорядочи.\n\n"
            f"Текущий документ:\n{state['document']}"
        )

    response = llm.invoke([
        SystemMessage(content=(
            "Ты — эксперт по структуре текста. "
            "Улучши документ: логика изложения, структура разделов, связность абзацев. "
            "Верни улучшенную версию документа целиком."
        )),
        HumanMessage(content=f"Задача: {state['task']}\n\n{prompt}")
    ])
    print(f"  [Структуралист, раунд {state['round'] + 1}] Обновлено ({len(response.content)} символов)")
    return {"document": response.content}


def factologist_node(state: RoundRobinState) -> dict:
    """Фактолог: проверяет и дополняет фактами."""
    response = llm.invoke([
        SystemMessage(content=(
            "Ты — эксперт по фактам. "
            "Проверь факты в документе, дополни конкретными данными и примерами. "
            "Верни улучшенную версию документа целиком."
        )),
        HumanMessage(content=f"Задача: {state['task']}\n\nДокумент:\n{state['document']}")
    ])
    print(f"  [Фактолог, раунд {state['round'] + 1}] Дополнено ({len(response.content)} символов)")
    return {"document": response.content}


def stylist_node(state: RoundRobinState) -> dict:
    """Стилист: улучшает стиль, убирает канцеляризмы."""
    response = llm.invoke([
        SystemMessage(content=(
            "Ты — эксперт по стилю. "
            "Улучши язык: убери канцеляризмы, сделай текст живым и читаемым. "
            "Верни улучшенную версию документа целиком."
        )),
        HumanMessage(content=f"Задача: {state['task']}\n\nДокумент:\n{state['document']}")
    ])
    print(f"  [Стилист, раунд {state['round'] + 1}] Отполировано ({len(response.content)} символов)")
    return {"document": response.content}

## Счётчик раундов и маршрутизация

После прохода всех трёх экспертов отдельный узел `counter` увеличивает счётчик раундов. Затем условное ребро `should_continue` решает: если лимит раундов достигнут — завершаем (`done`), иначе — новый круг (`next_round`) через тех же трёх экспертов.

Выделение счётчика в отдельный узел — осознанное решение: это отделяет бизнес-логику агентов от управляющей логики графа.

In [5]:
def round_counter(state: RoundRobinState) -> dict:
    """Увеличивает счётчик раундов."""
    new_round = state["round"] + 1
    print(f"  [Счётчик] Раунд {new_round} из {state['max_rounds']} завершён")
    return {"round": new_round}


def should_continue(state: RoundRobinState) -> str:
    """Проверка: продолжать раунды или завершить."""
    if state["round"] >= state["max_rounds"]:
        return "done"
    return "next_round"

## Сборка кругового графа

Граф строится как фиксированная цепочка: структуралист -> фактолог -> стилист -> счётчик -> проверка. Условное ребро из счётчика замыкает цикл обратно на структуралиста или завершает граф. Все рёбра между экспертами — безусловные, порядок прохода фиксирован.

In [6]:
graph = StateGraph(RoundRobinState)

graph.add_node("structuralist", structuralist_node)
graph.add_node("factologist", factologist_node)
graph.add_node("stylist", stylist_node)
graph.add_node("counter", round_counter)

# Круг: структуралист → фактолог → стилист → счётчик → проверка
graph.add_edge(START, "structuralist")
graph.add_edge("structuralist", "factologist")
graph.add_edge("factologist", "stylist")
graph.add_edge("stylist", "counter")

graph.add_conditional_edges("counter", should_continue, {
    "next_round": "structuralist",
    "done": END,
})

app = graph.compile()

## Запуск

Два раунда по три агента — шесть вызовов LLM. Каждый раунд улучшает документ по трём измерениям: структура, факты, стиль. Начальный документ — короткий черновик в одно предложение; к финалу это будет проработанный текст.

In [7]:
result = app.invoke(
    {
        "task": "Напиши обзор мультиагентных систем",
        "document": "Мультиагентные системы — это...",  # Начальный черновик
        "round": 0,
        "max_rounds": 2,
    },
    config={"recursion_limit": 25},
)

print(f"\n  Раундов пройдено: {result['round']}")
print(f"  Финальный документ ({len(result['document'])} символов):")
print(f"\n{result['document'][:500]}...")

  [Структуралист, раунд 1] Обновлено (6392 символов)


  [Фактолог, раунд 1] Дополнено (14655 символов)


  [Стилист, раунд 1] Отполировано (14083 символов)
  [Счётчик] Раунд 1 из 2 завершён


  [Структуралист, раунд 2] Обновлено (14178 символов)


  [Фактолог, раунд 2] Дополнено (16792 символов)


  [Стилист, раунд 2] Отполировано (16683 символов)
  [Счётчик] Раунд 2 из 2 завершён

  Раундов пройдено: 2
  Финальный документ (16683 символов):

# Обзор мультиагентных систем

## 1. Введение

Мультиагентные системы (Multi-Agent Systems, MAS) — это область искусственного интеллекта, распределённых вычислений и теории сложных систем, где несколько автономных агентов взаимодействуют друг с другом и со средой, чтобы добиться своих или общих целей. Под агентом обычно понимают программный или роботизированный компонент, который умеет **воспринимать среду, принимать решения и действовать** по своей стратегии поведения.

Интерес к мультиагентным...


## Итог

Round-Robin Discussion — паттерн кооперативной итеративной работы над одним артефактом. Ключевые особенности:

- **Один документ, три измерения**: каждый агент улучшает артефакт по своей оси (структура / факты / стиль), не разрушая вклад предыдущих
- **Кумулятивное улучшение**: каждый раунд наслаивает правки поверх предыдущих; два раунда дают заметно лучший результат, чем один
- **Фиксированный порядок**: в отличие от Supervisor, здесь нет динамической маршрутизации — порядок прохода определён заранее
- **Контроль стоимости**: `max_rounds` напрямую определяет количество вызовов LLM (раунды * количество агентов)